# DATATHON 2026 — The Gridbreakers
## Phần 1: Giải Câu hỏi Trắc nghiệm

Notebook này giải toàn bộ 10 câu MCQ bằng cách tính toán trực tiếp từ dữ liệu.

In [1]:
#Tự load dữ liệu, chạy local

In [2]:
import pandas as pd
import numpy as np

# -------------------------------------------------------
# Load dữ liệu
# -------------------------------------------------------
orders      = pd.read_csv('orders.csv', parse_dates=['order_date'])
products    = pd.read_csv('products.csv')
returns     = pd.read_csv('returns.csv')
web_traffic = pd.read_csv('web_traffic.csv')
order_items = pd.read_csv('order_items.csv', low_memory=False)
customers   = pd.read_csv('customers.csv')
payments    = pd.read_csv('payments.csv')
geography   = pd.read_csv('geography.csv')
sales       = pd.read_csv('sales.csv', parse_dates=['Date'])

print('Loaded all datasets successfully!')

Loaded all datasets successfully!


---
## Q1 — Trung vị số ngày giữa hai lần mua liên tiếp

**Đề bài:** Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu?

**Đáp án: C) 144 ngày**

In [3]:
# Sắp xếp đơn hàng theo khách hàng và ngày
orders_sorted = orders.sort_values(['customer_id', 'order_date'])

# Lọc chỉ khách hàng có >1 đơn hàng
multi_order = orders_sorted.groupby('customer_id').filter(lambda x: len(x) > 1)

# Tính gap giữa các đơn liên tiếp trong cùng 1 khách hàng
gaps = multi_order.groupby('customer_id')['order_date'].apply(
    lambda x: x.diff().dt.days.dropna()
)

all_gaps = gaps.explode().dropna().astype(float)
median_gap = all_gaps.median()

print(f'Tổng số gap tính được: {len(all_gaps):,}')
print(f'Median inter-order gap: {median_gap:.1f} ngày')
print('\n✅ Đáp án: C) 144 ngày')

Tổng số gap tính được: 556,699
Median inter-order gap: 144.0 ngày

✅ Đáp án: C) 144 ngày


---
## Q2 — Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất

**Đề bài:** Phân khúc sản phẩm nào có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức `(price − cogs) / price`?

**Đáp án: D) Standard**

In [4]:
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

margin_by_segment = (
    products.groupby('segment')['gross_margin']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'gross_margin': 'avg_gross_margin'})
)
margin_by_segment['avg_gross_margin'] = margin_by_segment['avg_gross_margin'].map('{:.4f}'.format)

print(margin_by_segment.to_string(index=False))
print('\n✅ Đáp án: D) Standard (gross margin ~31.3%)')

    segment avg_gross_margin
   Standard           0.3134
    Premium           0.2854
All-weather           0.2842
 Activewear           0.2656
Performance           0.2636
   Balanced           0.2580
     Trendy           0.2408
   Everyday           0.2363

✅ Đáp án: D) Standard (gross margin ~31.3%)


---
## Q3 — Lý do trả hàng phổ biến nhất với sản phẩm Streetwear

**Đề bài:** Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear, lý do trả hàng nào xuất hiện nhiều nhất?

**Đáp án: B) wrong_size**

In [5]:
streetwear_ids = products[products['category'] == 'Streetwear'][['product_id']]

streetwear_returns = returns.merge(streetwear_ids, on='product_id')

reason_counts = streetwear_returns['return_reason'].value_counts().reset_index()
reason_counts.columns = ['return_reason', 'count']

print(f'Tổng số lần trả hàng Streetwear: {len(streetwear_returns):,}')
print()
print(reason_counts.to_string(index=False))
print('\n✅ Đáp án: B) wrong_size (7,626 lần)')

Tổng số lần trả hàng Streetwear: 21,799

   return_reason  count
      wrong_size   7626
       defective   4330
not_as_described   3854
    changed_mind   3830
   late_delivery   2159

✅ Đáp án: B) wrong_size (7,626 lần)


---
## Q4 — Nguồn traffic có bounce_rate trung bình thấp nhất

**Đề bài:** Nguồn truy cập nào có tỷ lệ thoát trung bình thấp nhất?

**Đáp án: C) email_campaign**

In [6]:
bounce_by_source = (
    web_traffic.groupby('traffic_source')['bounce_rate']
    .mean()
    .sort_values()
    .reset_index()
    .rename(columns={'bounce_rate': 'avg_bounce_rate'})
)
bounce_by_source['avg_bounce_rate'] = bounce_by_source['avg_bounce_rate'].map('{:.6f}'.format)

print(bounce_by_source.to_string(index=False))
print('\n✅ Đáp án: C) email_campaign (avg bounce_rate thấp nhất)')

traffic_source avg_bounce_rate
email_campaign        0.004458
  social_media        0.004476
   paid_search        0.004478
      referral        0.004499
organic_search        0.004504
        direct        0.004511

✅ Đáp án: C) email_campaign (avg bounce_rate thấp nhất)


---
## Q5 — Tỷ lệ dòng order_items có áp dụng khuyến mãi

**Đề bài:** Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (promo_id không null)?

**Đáp án: C) 39%**

In [7]:
total_rows = len(order_items)
promo_rows = order_items['promo_id'].notna().sum()
pct = promo_rows / total_rows * 100

print(f'Tổng số dòng order_items: {total_rows:,}')
print(f'Số dòng có promo_id:       {promo_rows:,}')
print(f'Tỷ lệ:                     {pct:.1f}%')
print('\n✅ Đáp án: C) 39%')

Tổng số dòng order_items: 714,669
Số dòng có promo_id:       276,316
Tỷ lệ:                     38.7%

✅ Đáp án: C) 39%


---
## Q6 — Nhóm tuổi có số đơn hàng trung bình cao nhất

**Đề bài:** Nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất?

**Đáp án: A) 55+**

In [8]:
# Số đơn hàng mỗi khách hàng
orders_per_cust = orders.groupby('customer_id').size().reset_index(name='num_orders')

# Chỉ lấy customers có age_group != null
cust_nn = customers[customers['age_group'].notna()]

# Merge và tính trung bình
merged = cust_nn.merge(orders_per_cust, on='customer_id', how='left')
merged['num_orders'] = merged['num_orders'].fillna(0)

avg_orders = (
    merged.groupby('age_group')['num_orders']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
avg_orders['num_orders'] = avg_orders['num_orders'].map('{:.4f}'.format)

print(avg_orders.to_string(index=False))
print('\n✅ Đáp án: A) 55+ (avg ~5.41 đơn/khách hàng)')

age_group num_orders
      55+     5.4069
    45-54     5.3572
    35-44     5.3373
    25-34     5.2452
    18-24     5.2267

✅ Đáp án: A) 55+ (avg ~5.41 đơn/khách hàng)


---
## Q7 — Vùng tạo ra tổng doanh thu cao nhất

**Đề bài:** Vùng nào trong geography.csv tạo ra tổng doanh thu cao nhất?

> **Lưu ý:** sales.csv là dữ liệu tổng hợp không có thông tin vùng. Ta tính doanh thu theo vùng bằng cách join order_items → orders → geography.

**Đáp án: C) East**

In [9]:
# Gắn region vào orders qua zip
orders_geo = orders.merge(geography[['zip', 'region']], on='zip', how='left')

# Gắn region vào order_items
oi_region = order_items.merge(orders_geo[['order_id', 'region']], on='order_id', how='left')

# Doanh thu = quantity * unit_price - discount_amount
oi_region['line_revenue'] = (
    oi_region['quantity'] * oi_region['unit_price'] - oi_region['discount_amount']
)

revenue_by_region = (
    oi_region.groupby('region')['line_revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
revenue_by_region['line_revenue'] = revenue_by_region['line_revenue'].map('{:,.0f}'.format)

print(revenue_by_region.to_string(index=False))
print('\n✅ Đáp án: C) East (doanh thu cao nhất ~7.29 tỷ)')

 region  line_revenue
   East 7,291,150,819
Central 4,719,491,268
   West 3,670,227,178

✅ Đáp án: C) East (doanh thu cao nhất ~7.29 tỷ)


---
## Q8 — Phương thức thanh toán phổ biến nhất trong đơn bị huỷ

**Đề bài:** Trong các đơn hàng có order_status = 'cancelled', phương thức thanh toán nào được sử dụng nhiều nhất?

**Đáp án: A) credit_card**

In [10]:
cancelled = orders[orders['order_status'] == 'cancelled']

payment_counts = (
    cancelled['payment_method']
    .value_counts()
    .reset_index()
    .rename(columns={'count': 'num_orders'})
)

print(f'Tổng đơn bị huỷ: {len(cancelled):,}')
print()
print(payment_counts.to_string(index=False))
print('\n✅ Đáp án: A) credit_card (28,452 đơn)')

Tổng đơn bị huỷ: 59,462

payment_method  num_orders
   credit_card       28452
           cod       15468
        paypal        7817
     apple_pay        5190
 bank_transfer        2535

✅ Đáp án: A) credit_card (28,452 đơn)


---
## Q9 — Kích thước có tỷ lệ trả hàng cao nhất

**Đề bài:** Trong bốn kích thước S, M, L, XL — kích thước nào có tỷ lệ trả hàng cao nhất (số returns / số dòng order_items)?

**Đáp án: A) S**

In [11]:
sizes = ['S', 'M', 'L', 'XL']

# Số dòng order_items theo size
oi_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
oi_size = oi_size[oi_size['size'].isin(sizes)]
oi_count = oi_size.groupby('size').size().rename('oi_count')

# Số lần return theo size
ret_size = returns.merge(products[['product_id', 'size']], on='product_id', how='left')
ret_size = ret_size[ret_size['size'].isin(sizes)]
ret_count = ret_size.groupby('size').size().rename('ret_count')

result = pd.concat([oi_count, ret_count], axis=1)
result['return_rate'] = result['ret_count'] / result['oi_count']
result = result.sort_values('return_rate', ascending=False)
result['return_rate'] = result['return_rate'].map('{:.4f}'.format)

print(result.to_string())
print('\n✅ Đáp án: A) S (tỷ lệ trả hàng cao nhất ~5.65%)')

      oi_count  ret_count return_rate
size                                 
S       172042       9723      0.0565
L       173174       9741      0.0562
M       176428       9820      0.0557
XL      193025      10655      0.0552

✅ Đáp án: A) S (tỷ lệ trả hàng cao nhất ~5.65%)


---
## Q10 — Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất

**Đề bài:** Kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

**Đáp án: C) 6 kỳ**

In [12]:
# Lọc chỉ các kỳ trả góp hợp lệ theo đề bài: 1, 3, 6, 12
valid_plans = [1, 3, 6, 12]
payments_valid = payments[payments['installments'].isin(valid_plans)]

avg_payment = (
    payments_valid.groupby('installments')['payment_value']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'payment_value': 'avg_payment_value'})
)
avg_payment['avg_payment_value'] = avg_payment['avg_payment_value'].map('{:,.2f}'.format)

print(avg_payment.to_string(index=False))
print('\n✅ Đáp án: C) 6 kỳ (avg payment ~24,446)')

 installments avg_payment_value
            6         24,446.65
            3         24,399.64
           12         24,245.77
            1         24,113.27

✅ Đáp án: C) 6 kỳ (avg payment ~24,446)


---
## Tổng hợp đáp án

| Câu | Đáp án | Giá trị tính được |
|-----|--------|-------------------|
| Q1  | **C) 144 ngày** | Median inter-order gap = 144.0 ngày |
| Q2  | **D) Standard** | Gross margin trung bình = 31.3% |
| Q3  | **B) wrong_size** | 7,626 lần (nhiều nhất trong Streetwear) |
| Q4  | **C) email_campaign** | Avg bounce rate thấp nhất = 0.004458 |
| Q5  | **C) 39%** | 38.7% dòng có promo_id |
| Q6  | **A) 55+** | Avg 5.41 đơn/khách hàng |
| Q7  | **C) East** | Doanh thu ~7.29 tỷ (cao nhất) |
| Q8  | **A) credit_card** | 28,452 đơn bị huỷ |
| Q9  | **A) S** | Return rate = 5.65% |
| Q10 | **C) 6 kỳ** | Avg payment value = 24,446 |